<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 70 Classic

The notebook checks the classic look, the Windows 98 look of 3D-Baukasten 1.3 that the page switches
to: the windows float at the places `Form_Set` of `3D_bau.frm` computes for the screen, the four
views are squares of half the shorter side of Form1 as `ViewControl` makes them, the fields carry
the colours of 1.3, and the renderer draws black views with the white raster and wire bodies. Then
it checks the numbers of 1.3, the origin of the views, the register, the legend of GEOMETRIEDATEN,
the dialog Farben, the window F6, the Dia-Show and the splash, the chrome of the windows, the screen
of 800 by 600, the inverted views, the pan of the Schrägriss, the dialog Hilfe with the submenu
Beispiele, Command1 held down and the hatch of the frame FARBE. The page opens at 1024 by 768, the
resolution the thesis recommends, and section 16 opens a second page at 800 by 600.

A check names the ID of the row of `Documentation/quirks.md` it proves, and a code cell carries the
tag of the column Test. A check that `expect` makes waits for the last cell when it fails.

In [ ]:
width = 1024
height = 768
look = 'classic'

In [ ]:
import sys, json, math, re
sys.path.insert(0, '..')
from harness import *

In [ ]:
site = Site()
page = Browser(site.url, width=width, height=height, look=look)
page.open()
DEFERRED = []


def expect(condition, message, *context):
    """Checks as check() does, and keeps a failure for the last cell instead of stopping here."""
    try:
        check(condition, message, *context)
    except Failure as failure:
        DEFERRED.append(str(failure))
        print(f"FAIL {message}")


def fresh(look=None):
    """Empties the scene through Datei › Neu and answers Ja, as a user does."""
    page.eval("() => { bausteine.act('neu'); }")
    page.settle()
    if page.eval("() => document.querySelector('#msgbox').open"):
        page.page.click('#msgbox .c-default')
        page.settle()
    page.key('Escape')
    page.key('Escape')


check(page.errors() == [], 'the page starts without an error', page.errors())


def at(view, x, y, z):
    """Answers the pixel of the page that shows the point (x, y, z) in the view `view`."""
    return page.eval(
        "([v, x, y, z]) => { const r = document.querySelector(`.view[data-view='${v}']`).getBoundingClientRect();"
        " const b = bausteine.app.bridge; const p = b.project(v, x, y, z); if (!p) return null;"
        " const w = b.cameras ? b.cameras[v * 18 + 16] : 0; const inset = w > 0 ? Math.max(0, (r.width - w) / 2) : 0;"
        " return [r.left + inset + p[0], r.top + inset + p[1]]; }", [view, x, y, z])


def hover_world(view, x, y, z):
    """Moves the pointer over the point (x, y, z) of the view `view`."""
    p = at(view, x, y, z)
    page.page.mouse.move(p[0], p[1])
    page.settle()
    return p


def white_near(view, x, y, z, r=2):
    """Answers the number of white pixels of the picture in a square of 2r around the point (x, y, z) of `view`."""
    p = at(view, x, y, z)
    return page.sample_box(p[0] - r, p[1] - r, 2 * r, 2 * r).get('#ffffff', 0)

## 1. The places of Form_Set

At a screen of W by H pixels `Form_Set` puts F3 at 0, 0 with 48 by 500, F2 under it down to H − 27,
the strip `StatusH` of the task bar of Windows 98, Form1 at 48, 0 with W − 300 by H − 50, F4 at W −
252, 0 with 252 by 580, F6 under F4 at a top of 580 with the outer size of the record in the EXE,
252 by 110, and F5 at the bottom of Form1 in its width. `Form_Set` gives the frame of the status the
width of F5 and ignores its left, so the frame runs past the right edge of F5, which cuts it off.
The virtual screen of the classic look is 1024 by 768 by default.

In [ ]:
W, H = width, height
expected = {
    '#f3': (0, 0, 48, 500),
    '#f2': (0, 500, 48, H - 527),
    '#form1': (48, 0, W - 300, H - 50),
    '#f4': (W - 252, 0, 252, 580),
    '#f6': (W - 252, 580, 252, 110),
}
for selector, box in expected.items():
    got = page.rect(selector)
    check(all(abs(a - b) <= 2 for a, b in zip(got, box)), f'T-p74a {selector} sits where Form_Set puts it', got, box)
check(page.eval("() => ['#form1', '#f2', '#f3', '#f4', '#f5', '#f6'].every((s) => getComputedStyle(document.querySelector(s)).position === 'absolute')"), 'T-p74a the windows float on the virtual screen')
check(abs(page.rect('#f2')[1] + page.rect('#f2')[3] - (H - 27)) <= 1, 'Q-12 F2 ends 27 px above the bottom, the strip StatusH keeps for the task bar', page.rect('#f2'))
f5 = page.rect('#f5')
check(abs(f5[0] - 48) <= 2 and abs(f5[2] - (W - 300)) <= 2 and abs(f5[3] - 35) <= 2, 'F5 is 35 pixels high in the width of Form1', f5)
frame = page.rect('#f5-frame6')
check(frame[0] + frame[2] > f5[0] + f5[2] + 100 and page.css('#f5', 'overflow') == 'hidden', 'Q-13 the frame of the status runs past the right edge of F5, which cuts it off', frame, f5)
expect(page.eval("() => bausteine.app.settings.screen") == '1024x768', 'T-p74a the virtual screen of the classic look is 1024 by 768 by default', page.eval("() => bausteine.app.settings.screen"))

## 2. The views of ViewControl

Each view is a square of F / 2, where F is the shorter side of the client of Form1, the Kreuzriss at
the top left, the Aufriss beside it, the Schrägriss under the Kreuzriss and the Grundriss under the
Aufriss.

In [ ]:
v = {n: page.rect(f'.view[data-view="{n}"]') for n in range(4)}
side = v[1][2]
check(all(abs(b[2] - side) <= 1 and abs(b[3] - side) <= 1 for b in v.values()), 'four squares of one size', v)
check(abs(v[0][0] - (v[1][0] + side)) <= 1 and abs(v[3][1] - (v[1][1] + side)) <= 1 and abs(v[2][0] - v[0][0]) <= 1 and abs(v[2][1] - v[3][1]) <= 1, 'the quadrants of ViewControl', v)
views = page.rect('#views')
check(abs(side - min(views[2], views[3]) // 2) <= 1, 'Q-17 the side is half the shorter side of the client area, so no view is cut off', side, views)
check(v[2][1] + v[2][3] <= views[1] + views[3] + 1, 'Q-17 the lower row ends inside the client of Form1', v[2], views)

## 3. The colours of 1.3

The fields x, y and z of F5 have red, yellow and green digits on black, the zoom and the status
white on the blue `&H00C00000&`, the fields of the position and the size yellow on black, the turns
red, the frame GEOMETRIEDATEN is teal, and the active caption purple, the scheme of most screenshots
of the thesis in its true colours. The label `x` is black, where 1.3 lost the high bit of its system
colour.

In [ ]:
color = lambda s, p='color': page.eval("([s, p]) => getComputedStyle(document.querySelector(s))[p]", [s, p])
check(color('#tx') == 'rgb(255, 0, 0)' and color('#ty') == 'rgb(255, 255, 0)' and color('#tz') == 'rgb(0, 255, 0)', 'red, yellow and green digits', color('#tx'), color('#ty'), color('#tz'))
check(color('#tx', 'backgroundColor') == 'rgb(0, 0, 0)', 'on black', color('#tx', 'backgroundColor'))
check(color('#tstatus', 'backgroundColor') == 'rgb(0, 0, 192)' and color('#tzoom', 'backgroundColor') == 'rgb(0, 0, 192)', 'zoom and status on the blue of 1.3', color('#tstatus', 'backgroundColor'))
check(color('#t1-1') == 'rgb(255, 255, 0)' and color('#t1-7') == 'rgb(255, 0, 0)', 'yellow position, red turn', color('#t1-1'), color('#t1-7'))
check(color('#frame1', 'backgroundColor') in ('rgb(0, 128, 128)', 'rgb(0, 192, 192)'), 'the teal frame GEOMETRIEDATEN', color('#frame1', 'backgroundColor'))
check(page.css('.c-win.active > .win-title', 'background-color') == '#800080' and page.css('body', 'background-color') == '#008080', 'T-p92c the purple caption and the teal desktop of the thesis in their true colours', page.css('.c-win.active > .win-title', 'background-color'))
expect(page.css('#lx', 'color') == '#000000', 'Q-91 the label x of F5 is black', page.css('#lx', 'color'))
zoom = page.eval("() => [document.querySelector('#tzoom').value, bausteine.app.zoomf]")
check(zoom[0] == zoom_text(zoom[1]), 'the 2D zoom shows the raw zoomf / 10 of 1.3 with a comma', zoom)
page.page.click('#d2-zoomout'); page.settle()
page.page.click('#d2-zoomout'); page.settle()
zoom = page.eval("() => [document.querySelector('#tzoom').value, bausteine.app.zoomf]")
check(zoom[0] == zoom_text(zoom[1]) and ',' in zoom[0], 'and after two steps down the ladder', zoom)
page.page.click('#d2-zoomin'); page.page.click('#d2-zoomin'); page.settle()

## 4. The tool buttons and their icons

The buttons of F2 and F3 are 39 by 38 pixels with a picture of 32 by 32 in the sixteen colours of
the Windows palette, the places 10 to 13 and the bodies Disc and Ebene included. The I-beam of the
icon Position is dark grey, where 1.3 drew it with inverting pixels.

In [ ]:
sizes = page.eval("() => [...document.querySelectorAll('#tb1 .tb-button, #tb2 .tb-button')].map((b) => { const r = b.getBoundingClientRect(); return [Math.round(r.width), Math.round(r.height)]; })")
check(all(s == [39, 38] for s in sizes), 'every button is 39 by 38', sizes)
palette = {'#000000', '#800000', '#008000', '#808000', '#000080', '#800080', '#008080', '#c0c0c0', '#808080', '#ff0000', '#00ff00', '#ffff00', '#0000ff', '#ff00ff', '#00ffff', '#ffffff'}
fills = lambda selector: [f.lower() for f in page.texts(f'{selector} svg [fill]', 'fill')]
outside = sorted({f for f in fills('#tb1') + fills('#tb2') if f not in palette and f != 'none'})
check(fills('#tb1') and outside == [], 'the icons use the sixteen colours alone', outside)
slots = [fills(f'#tb1 [data-tool="{n}"]') for n in range(10, 14)]
check(all(s and set(s) <= palette for s in slots), 'Q-77 the places 10 to 13 hold pixel icons in the sixteen colours of the VGA palette', slots)
bodies = [fills(f'#tb2 [data-body="{b}"]') for b in ('Disc', 'Ebene')]
check(all(len(set(b)) > 2 and set(b) <= palette for b in bodies), 'Q-68 Disc and Ebene get icons in the VGA palette', bodies)
check('#808080' in fills('#tb1 [data-tool="1"]'), 'C-19 the icon Position draws its I-beam dark grey', sorted(set(fills('#tb1 [data-tool="1"]'))))

## 5. The drawing of 1.3

The renderer draws the views black with a white dot at every unit and the bodies as wire models in
their colours. The picture of the Kreuzriss is read back from the worker, every pixel of it.

In [ ]:
path, stats = page.screenshot('70_classic')
r = page.rect('.view[data-view="1"]')
histogram = page.sample_box(r[0] + 4, r[1] + 4, r[2] - 8, r[3] - 8)
total = sum(histogram.values())
black = histogram.get('#000000', 0) / total
white = histogram.get('#ffffff', 0) / total
check(black > 0.8, 'the Kreuzriss is black', black)
check(white > 0.001, 'with the white raster', white)
check(1 - black - white > 0.0005, 'and the coloured wires of the bodies', list(histogram.items())[:6])

## 6. The password of the tab Einstellung

The field shows a `#` for every character, `PasswordChar` of `F4.frm`, and 2002 unlocks the paths.

In [ ]:
page.page.click('#sstab1 [data-tab="einstellung"]'); page.settle()
page.page.click('#tpasswd')
page.page.keyboard.type('2002')
check(page.eval("() => document.querySelector('#tpasswd').value") == '####', 'C-16 the field shows # for each character', page.eval("() => document.querySelector('#tpasswd').value"))
check(page.eval("() => document.querySelector('#tbau-home').disabled") is False, '2002 unlocks the paths')
page.page.click('#bdatei-def'); page.settle()
page.page.click('#sstab1 [data-tab="koerper"]')

## 7. Back to the modern look

The switch works both ways, the views follow the look, and no window keeps a place of the classic
look. The classic look flips eleven named toggles of presentation together and lists them on the
root element, the modern look lists none, and no data differ between the looks. The dark theme of
the renderer applies to the modern look only, so its entry in the menu Ansicht stands unchecked and
disabled in the classic look and checked again in the modern look.

In [ ]:
TOGGLES = ['raw-zoom', 'empty-status', 'label-clip', 'unscaled-slide', 'no-orbit', 'snap-whole', 'classic-captions', 'cavalier', 'origin-13', 'windows', 'vga-icons']
toggles = lambda: (page.eval("() => document.documentElement.getAttribute('data-look-toggle')") or '').split()
items = page.eval("() => JSON.stringify(bausteine.app.scene.items)")
check(toggles() == TOGGLES, 'R-47 the classic look lists its eleven toggles on the root element', toggles())
page.eval("() => bausteine.setLook('modern')")
page.settle()
check(page.eval("() => document.documentElement.dataset.look") == 'modern', 'the modern look')
check(page.eval("() => document.querySelector('#form1').style.left") == '', 'no window keeps an inline place')
check(toggles() == [] and page.eval("() => JSON.stringify(bausteine.app.scene.items)") == items, 'R-47 the modern look lists none, and the data are the same', toggles())
MARK = "() => { const e = document.querySelector('#menubar [data-entry=dark]'); return [e.getAttribute('aria-checked'), e.disabled]; }"
CLOSED = "() => [...document.querySelectorAll('#menubar .menu-popup')].every((p) => p.hidden || getComputedStyle(p).display === 'none')"
page.eval("() => bausteine.act('dark')"); page.settle()
marks = [page.eval(MARK)]
page.look('classic')
marks.append(page.eval(MARK))
page.look('modern')
marks.append(page.eval(MARK))
page.eval("() => bausteine.act('dark')"); page.settle()
marks.append(page.eval(MARK))
check(marks == [['true', False], ['false', True], ['true', False], ['false', False]] and page.eval(CLOSED), 'R-60 the entry of the dark theme is unchecked and disabled in the classic look and checked again in the modern look, with no menu opened', marks)
page.eval("() => bausteine.setLook('classic')")
page.settle()
page.eval("() => bausteine.act('dark')"); page.settle()
r = page.rect('.view[data-view="1"]')
histogram = page.sample_box(r[0] + 4, r[1] + 4, r[2] - 8, r[3] - 8)
check(page.eval("() => bausteine.app.dark") is True and histogram.get('#000000', 0) / sum(histogram.values()) > 0.8, 'R-60 the classic views stay black in the dark theme', list(histogram.items())[:3])
page.eval("() => bausteine.act('dark')"); page.settle()
check(page.errors() == [], 'no error on the way', page.errors())

## 8. The numbers of 1.3

Timer1 of 1.3 writes the raw Single `zoomf / 10` into the field of the 2D zoom, with the comma of
the German locale and no rounding: `1,2` at the start, `1,44`, `1,728` and `2,0736` after three
steps in, and on the way down `1` for the Single 1 and `0,8333333` one step under the start. The
fields of tab Körper accept the comma and the point and show the comma.

In [ ]:
fresh()
while abs(page.eval("() => bausteine.app.zoomf") - 12) > 1e-6:
    page.page.click('#d2-zoomin' if page.eval("() => bausteine.app.zoomf") < 12 else '#d2-zoomout'); page.settle()
texts = [page.eval("() => document.querySelector('#tzoom').value")]
for _ in range(3):
    page.page.click('#d2-zoomin'); page.settle()
    texts.append(page.eval("() => document.querySelector('#tzoom').value"))
check(texts == ['1,2', '1,44', '1,728', '2,0736'], 'Q-31 the classic bar shows the raw zoom with a comma', texts)
down = []
for _ in range(5):
    page.page.click('#d2-zoomout'); page.settle()
    down.append(page.eval("() => document.querySelector('#tzoom').value"))
check(down == ['1,728', '1,44', '1,2', '1', '0,8333333'], 'Q-31 and down again to 0,8333333 one step under the start', down)
page.page.click('#d2-zoomin'); page.page.click('#d2-zoomin'); page.settle()
check(page.eval("() => bausteine.app.zoomf") == 12, 'back at the start of the ladder')
page.page.click('#tb2 [data-body="Würfel"]'); page.settle()
page.page.fill('#t1-1', '0.5'); page.page.fill('#t1-2', '1,5'); page.page.fill('#t1-4', '2,5')
page.page.click('#btn-einf'); page.settle()
item = page.eval("() => bausteine.app.scene.items[0]")
check(item['position']['x'] == 0.5 and item['position']['y'] == 1.5 and item['size']['x'] == 2.5, 'Q-44 the fields accept the point and the comma', item['position'], item['size'])
page.page.click('#cb2 [data-line="1"]'); page.settle()
check(page.fields()[:2] == ['0,5', '1,5'] and page.fields()[3] == '2,5', 'Q-44 and show the comma of German', page.fields())

## 9. The origin of the views

1.3 computes the origin as F / 5 pixels and applies it in the user scale 328.27 of its picture
boxes, which puts it at the fraction (F / 5) / 328.27 of each view, 43.7 percent at 1024 by 768 with
F the shorter outer side of Form1. It lies that far from the edge each positive axis points away
from: the right and the bottom edge of the Kreuzriss, the left and the bottom edge of the Aufriss,
the left and the top edge of the Grundriss. The axes run from −2 to 10 units, and the dots of the
raster are single white pixels, as `PSet` of 1.3 drew them.

In [ ]:
F = min(page.rect('#form1')[2:])
fraction = (F / 5) / 328.27
check(abs(fraction - 0.4374) < 0.0005, 'the fraction of 1.3 at 1024 by 768', F, fraction)
cameras = page.eval("() => Array.from(bausteine.app.bridge.cameras)")
for view, flips in ((SIDE, (True, True)), (FRONT, (False, True)), (TOP, (False, False))):
    w, h = cameras[view * 18 + 16], cameras[view * 18 + 17]
    p = page.eval(f"() => bausteine.app.bridge.project({view}, 0, 0, 0)")
    dx = w - p[0] if flips[0] else p[0]
    dy = h - p[1] if flips[1] else p[1]
    check(abs(dx - fraction * w) <= 1 and abs(dy - fraction * h) <= 1, f'Q-20 the origin of the view {view} lies at 43.7 percent from its edges', p, w, h)
ends = {}
for name, view, axis in (('Aufriss y', FRONT, 1), ('Aufriss z', FRONT, 2), ('Kreuzriss x', SIDE, 0), ('Kreuzriss z', SIDE, 2), ('Grundriss x', TOP, 0), ('Grundriss y', TOP, 1)):
    point = lambda t: [t if i == axis else 0 for i in range(3)]
    ends[name] = [white_near(view, *point(t)) > 0 for t in (-2.75, -1.75, 9.75, 10.75)]
check(all(e == [False, True, True, False] for e in ends.values()), 'Q-20 the axes run from -2 to 10 units', ends)
r = page.rect('.view[data-view="2"]')
histogram = page.sample_box(r[0] + 4, r[1] + 4, r[2] - 8, r[3] - 8)
grey = sum(n for c, n in histogram.items() if c not in ('#000000', '#ffffff') and c[1:3] == c[3:5] == c[5:7])
expect(grey <= histogram.get('#ffffff', 0) / 10, 'the dots of the raster are single white pixels, not grey', list(histogram.items())[:6])

## 10. The register in the classic look

The classic register keeps the widths of the buttons of 1.3, draws CB1 bold, clips the labels at the
widths of 1.3, takes the keyboard through the controls in the order of the layout, and keeps the
captions `Frame6`, `Command1` and `Command2` of tab Kamera with the labels Zoom, x, y and z beside
the sliders. Where tab Bilder has its hidden text box, a small button `…` of 24 by 16 adds pictures,
and tab Oberfläche stays a page of its own, as empty as in 1.3.

In [ ]:
widths = lambda ids: [round(page.rect(i)[2]) for i in ids]
page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
check(widths(['#btn-einf', '#btn-change', '#btn-clear']) == [65, 65, 57], 'F4-Q15 Löschen keeps its 855 twips on tab Körper', widths(['#btn-einf', '#btn-change', '#btn-clear']))
page.page.click('#sstab1 [data-tab="ausgabe"]'); page.settle()
check(widths(['#bpov-comp', '#bpov-save']) == [89, 85], 'F4-Q15 Ausgeben keeps its 1275 twips', widths(['#bpov-comp', '#bpov-save']))
page.page.click('#sstab1 [data-tab="einstellung"]'); page.settle()
check(widths(['#bdatei-change', '#bdatei-def']) == [89, 81], 'F4-Q15 Standard keeps its 1215 twips', widths(['#bdatei-change', '#bdatei-def']))
page.page.click('#sstab1 [data-tab="bilder"]'); page.settle()
cmdia, frame, tab = page.rect('#cmdia'), page.rect('#frame9-1'), page.rect('[data-page="bilder"]')
check([round(cmdia[2]), round(cmdia[3])] == [24, 16] and abs(cmdia[1] - frame[1]) <= 1 and [round(cmdia[0] - tab[0]), round(cmdia[1] - tab[1])] == [64, 408] and page.texts('#cmdia') == ['…'], 'F4-Q11 a button … of 24 by 16 at 64, 408 adds pictures', cmdia, tab)
page.page.click('#sstab1 [data-tab="oberflaeche"]'); page.settle()
shown = page.eval("() => [...document.querySelectorAll('[data-page=oberflaeche] *')].filter((e) => getComputedStyle(e).display !== 'none').length")
front = page.eval("() => { const p = document.querySelector('[data-page=oberflaeche]'); return !p.hidden && getComputedStyle(p).display !== 'none' && document.querySelector('#sstab1 [data-tab=oberflaeche]').getAttribute('aria-selected') === 'true'; }")
check(front and shown == 0, 'F4-Q1 tab Oberfläche stays a page of its own, empty in the classic look as in 1.3', shown)
page.page.click('#sstab1 [data-tab="kamera"]'); page.settle()
check(page.texts('#frame6 > legend') == ['Frame6'] and page.texts('#start') == ['Command1'] and page.texts('#command2') == ['Command2'], 'F4-Q3 the classic look shows the captions of 1.3', page.texts('#frame6 > legend'))
check(page.texts('.hs-label') == ['Zoom', 'x', 'y', 'z'] and all(page.visible(s) for s in ('#lhs-4', '#lhs-1', '#lhs-2', '#lhs-3')), 'F4-Q8 the labels Zoom, x, y and z stand beside the sliders', page.texts('.hs-label'))
page.page.fill('#hst-3', '4'); page.page.press('#hst-3', 'Enter')
page.page.click('#start'); page.until("() => document.querySelector('#start').getAttribute('aria-pressed') === 'false'")
check(abs(page.eval("() => bausteine.app.turn.z") - 4) < 1e-9, 'Command1 still starts the turn', page.eval("() => bausteine.app.turn"))
page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
page.entry('Quader')
check(page.css('#cb1-edit', 'font-weight') == '700', 'C-9 the classic look draws CB1 bold', page.css('#cb1-edit', 'font-weight'))
label = page.eval("() => { const e = document.querySelector('#l1-4'); const s = getComputedStyle(e); return [Math.round(e.getBoundingClientRect().width), s.overflow, s.whiteSpace, e.scrollWidth > e.clientWidth]; }")
check(label[:3] == [69, 'hidden', 'nowrap'], 'F4-Q25 the label of the size clips at the 69 px of 1.3 on one line', label)
page.page.focus('#cb2')
order = []
for _ in range(4):
    page.page.keyboard.press('Tab')
    order.append(page.eval("() => document.activeElement.id"))
check(order == ['cb1-edit', 't1-1', 't1-2', 't1-3'], 'F4-Q19 the keyboard takes the controls in the order of the layout', order)

## 11. The legend of GEOMETRIEDATEN

The frame is teal `#008080` by default, as three of the four pictures of the thesis show, and
Bildschirmeinstellungen switch it to the `#00C0C0` of `F4.frm`. The legend is white on teal and
black on `#00C0C0`, and the labels stay black on both.

In [ ]:
def geo(colour):
    """Chooses the colour of the frame GEOMETRIEDATEN in Bildschirmeinstellungen."""
    page.eval("() => { bausteine.act('bildschirm'); }"); page.settle()
    page.page.select_option('#screen-dialog select[name="geo"]', colour)
    page.page.click('#screen-dialog button[value="ok"]'); page.settle()
    return [page.css('#frame1', 'background-color'), page.css('#frame1 > legend', 'color'), page.css('#l1-1', 'color')]


check([page.css('#frame1', 'background-color'), page.css('#frame1 > legend', 'color')] == ['#008080', '#ffffff'], 'C-8 the frame is teal by default, R-39 its legend white', page.css('#frame1 > legend', 'color'))
cyan = geo('#00c0c0')
check(cyan == ['#00c0c0', '#000000', '#000000'], 'R-39 on #00C0C0 the legend turns black, and the labels stay black', cyan)
teal = geo('#008080')
check(teal == ['#008080', '#ffffff', '#000000'], 'C-8 the setting switches back to teal with the white legend and black labels', teal)

## 12. The dialog Farben

The dialog of Windows has the purple caption with `?` and `×`, the 48 basic colours, the 16 custom
colours of the grey wedge it starts with, and `Farben definieren >>`, which is enabled and opens the
panel of hue, saturation and lightness as Windows counts them, 0 to 239 and 0 to 240, beside red,
green and blue.

In [ ]:
page.page.click('#cb2 [data-line="1"]'); page.settle()
page.page.click('#bcolor'); page.settle()
chrome = page.eval("() => [document.querySelector('#farben').open, !!document.querySelector('#farben .win-help'), !!document.querySelector('#farben .win-close')]")
check(chrome == [True, True, True] and page.css('#farben .win-title', 'background-color') == '#800080', 'the dialog with its purple caption, ? and ×', chrome)
check(page.eval("() => [...document.querySelectorAll('#farben .farben-label')].map((l) => l.innerHTML)") == ['<u>G</u>rundfarben:', '<u>B</u>enutzerdefinierte Farben:'], 'T-p39b the labels of Windows with their hot letters')
check(page.eval("() => [document.querySelectorAll('.farben-grund .farben-swatch').length, document.querySelectorAll('.farben-custom .farben-swatch').length]") == [48, 16], 'T-p39b 48 basic and 16 custom colours')
wedge = page.texts('.farben-custom .farben-swatch', 'data-color')
check(wedge == [f'#{v:02X}{v:02X}{v:02X}' for v in range(255, 0, -16)], 'T-p93f the custom colours start with the grey wedge', wedge)
check(page.eval("() => document.querySelector('#farben-define').disabled") is False and page.eval("() => document.querySelector('.farben-full').hidden"), 'C-10 Farben definieren is enabled')
page.page.click('#farben-define'); page.settle()
limits = page.eval("() => ['#farben-h', '#farben-s', '#farben-l', '#farben-r', '#farben-g', '#farben-b'].map((s) => document.querySelector(s).max)")
check(not page.eval("() => document.querySelector('.farben-full').hidden") and limits == ['239', '240', '240', '255', '255', '255'], 'C-10 and opens the panel of hue, saturation and lightness', limits)
check(not page.visible('#farben-picker'), 'the free picker of the modern look stays hidden')
page.page.click('#farben-cancel'); page.settle()

## 13. The window F6

F6 takes the record of the EXE: 252 px wide outside with a client of 244 by 86, the frame TEXTUR at
0, 0 with 241 by 49 holding CB3 at 8, 16 with 225 by 21, and the frame FARBE at 0, 48 with 241 by 37
holding the button Farbe at 88, 8 with 65 by 25. The frame FARBE is grey `#e0e0e0` until a body is
selected and takes its colour then.

In [ ]:
client = page.rect('#f6 > .win-client')
check([round(client[2]), round(client[3])] == [244, 86] and round(page.rect('#f6')[2]) == 252, 'R-38 F6 is 252 px wide with a client of 244 by 86', page.rect('#f6'), client)
rel = lambda s, base: [round(a - b) for a, b in zip(page.rect(s)[:2], base[:2])] + [round(x) for x in page.rect(s)[2:]]
places = {s: rel(s, client) for s in ('#frame11', '#cb3', '#frame-farbe')}
places['#bcolor'] = rel('#bcolor', page.rect('#frame-farbe'))
check(places == {'#frame11': [0, 0, 241, 49], '#cb3': [8, 16, 225, 21], '#frame-farbe': [0, 48, 241, 37], '#bcolor': [88, 8, 65, 25]}, 'Q-80 the frames TEXTUR and FARBE and their controls at the places of the record', places)
page.key('Escape'); page.key('Escape')
page.page.click('#cb2 [data-line="0"]'); page.settle()
check(page.css('#frame-farbe', 'background-color') == '#e0e0e0', 'R-38 the frame FARBE is grey while no body is selected')
page.page.click('#cb2 [data-line="1"]'); page.settle()
colour = page.eval("() => bausteine.app.scene.items[0].color").lower()
check(page.css('#frame-farbe', 'background-color') == colour, 'R-38 and takes the colour of the selected body', colour)

## 14. The Dia-Show and the splash

The show of F9 fills the virtual screen in black and draws the picture unscaled at the top left. Its
three buttons `<`, `>` and `End` keep their captions and their order at the bottom right inside the
screen, F2 and F3 hide while it shows, Enter ends it and Esc, the arrows and Space work, and End
leaves the windows where they are. Hilfe › Über shows the card of the splash with factual lines, the
title `3D-Bausteine`, one line `Version 2.0` and a logo drawn as SVG, and any click or key closes
it.

In [ ]:
import tempfile
from PIL import Image
folder = tempfile.mkdtemp(prefix='bausteine-dia-')
pictures = []
for name, colour in (('rot.png', (255, 0, 0)), ('blau.png', (0, 0, 255))):
    pictures.append(f'{folder}/{name}')
    Image.new('RGB', (300, 200), colour).save(pictures[-1])
page.page.click('#sstab1 [data-tab="bilder"]'); page.settle()
kept = page.eval("() => document.querySelectorAll('#file1 [data-index]').length")
page.page.set_input_files('#cmdia-file', pictures)
page.until("(n) => document.querySelectorAll('#file1 [data-index]').length >= n", kept + len(pictures))
page.page.click('#file1 [data-index="0"]'); page.settle()
before = {s: page.rect(s) for s in ('#form1', '#f4', '#f5', '#f6')}
LOADED = "() => { const i = document.querySelector('#f9-picture'); return !document.querySelector('#f9').hidden && i.complete && i.naturalWidth > 0; }"
page.page.click('#bbild-anz'); page.until(LOADED)
check(page.rect('#f9') == [0, 0, width, height] and page.css('#f9', 'background-color') == '#000000', 'F9 fills the virtual screen in black', page.rect('#f9'))
check(page.rect('#f9-picture') == [0, 0, 300, 200], 'Q-99 the classic look draws the picture unscaled at the top left', page.rect('#f9-picture'))
buttons = [page.rect(s) for s in ('#bbild-rueck', '#bbild-weiter', '#bbild-ende')]
check(page.texts('.f9-buttons button') == ['<', '>', 'End'] and all(b[0] + b[2] <= width and b[1] + b[3] <= height and b[0] > width / 2 and b[1] > height / 2 for b in buttons) and buttons[0][0] < buttons[1][0] < buttons[2][0], 'Q-98 the three buttons keep their captions and their order at the bottom right inside the screen', buttons)
check(not page.visible('#f2') and not page.visible('#f3'), 'Q-100 the bars hide while the show is open')
SHOWN = "() => { const i = document.querySelector('#f9-picture'); const c = document.createElement('canvas'); c.width = c.height = 1; const g = c.getContext('2d'); g.drawImage(i, i.naturalWidth / 2, i.naturalHeight / 2, 1, 1, 0, 0, 1, 1); const d = g.getImageData(0, 0, 1, 1).data; return d[0] > 128 ? 'rot' : d[2] > 128 ? 'blau' : 'anders'; }"
SHOWS = "(name) => (" + SHOWN + ")() === name"
check(page.eval(SHOWN) == 'rot', 'the show starts with the first picture')
page.key('ArrowRight'); page.until(SHOWS, 'blau')
check(page.visible('#f9') and page.eval(SHOWN) == 'blau', 'Q-98 the arrow shows the next picture')
page.key('ArrowLeft'); page.until(SHOWS, 'rot')
check(page.eval(SHOWN) == 'rot', 'Q-98 and the other arrow the one before')
page.key(' '); page.until(SHOWS, 'blau')
check(page.visible('#f9') and page.eval(SHOWN) == 'blau', 'Q-98 Space shows the next picture')
page.key('Enter'); page.settle()
check(not page.visible('#f9') and page.visible('#f2') and page.visible('#f3'), 'Q-98 Enter ends the show, Q-100 and the bars come back')
page.page.click('#bbild-anz'); page.settle()
page.key('Escape'); page.settle()
check(not page.visible('#f9'), 'Q-98 Esc ends the show')
page.page.click('#bbild-anz'); page.settle()
page.page.click('#bbild-ende'); page.settle()
check(not page.visible('#f9') and {s: page.rect(s) for s in before} == before, 'Q-101 End closes the show and leaves the windows where they are')
page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
page.eval("() => bausteine.act('info')"); page.settle()
card = page.texts('.splash-card h1, .splash-card p')
check(card[:4] == ['3D-Bausteine', 'Version 2.0', 'nach 3D-Baukasten 1.3 von Herbert Kleber (2002)', 'Florian Kleber 2026'], 'Q-104 the card shows factual lines only', card)
check(page.texts('.splash-title') == ['3D-Bausteine'], 'Q-105 the title reads 3D-Bausteine')
check([t for t in card if 'Version' in t] == ['Version 2.0'], 'Q-106 one line reads Version 2.0', card)
logo = page.eval("() => { const m = document.querySelector('.splash-logo svg').getScreenCTM(); return [m.a, m.d]; }")
check(abs(logo[0] - logo[1]) < 1e-6, 'Q-108 the logo is an SVG drawing that scales evenly across and down', logo, page.rect('.splash-logo svg'))
page.page.click('.splash-title'); page.settle()
check(page.eval("() => document.querySelector('#splash').hidden"), 'Q-109 a click on a label closes the card')
page.eval("() => bausteine.act('info')"); page.settle()
page.key('x'); page.settle()
check(page.eval("() => document.querySelector('#splash').hidden"), 'Q-109 and so does any key')

## 15. The captions and the chrome

Form1 carries `3D-Baukasten` with the name of the scene and the buttons `_`, `▭` and `×`, the tool
windows carry only `×` and the captions `3D`, an empty one, `Eingaben` and `Farben/Texturen`, and F5
has no title bar and cannot be dragged. F2 and F3 stay above the other windows. A window dragged by
its caption keeps its place for the session, Fenster anordnen and a reload put it back where
`Form_Set` puts it, and the × of a window hides it until Alle Fenster zeigen brings it back with its
state.

In [ ]:
chrome = page.eval('''() => Object.fromEntries(['#form1', '#f2', '#f3', '#f4', '#f5', '#f6'].map((s) => {
  const t = document.querySelector(s).querySelector(':scope > .win-title');
  return [s, t ? [t.querySelector('.win-caption').textContent, [...t.querySelectorAll('.win-buttons > *')].filter((b) => getComputedStyle(b).display !== 'none').map((b) => b.className)] : null];
}))''')
check(chrome['#form1'][0].startswith('3D-Baukasten - ') and chrome['#form1'][1] == ['win-min', 'win-max', 'win-close'], 'Form1 carries 3D-Baukasten with the name of the scene and three buttons', chrome['#form1'])
check(chrome['#f2'][0] == '3D' and chrome['#f3'][0] == '' and chrome['#f4'][0] == 'Eingaben' and chrome['#f6'][0] == 'Farben/Texturen', 'Q-67 the classic caption of F2 is 3D, beside an empty one, Eingaben and Farben/Texturen', chrome)
check(all(chrome[s][1] == ['win-close'] for s in ('#f2', '#f3', '#f4', '#f6')), 'the tool windows carry only ×')
check(chrome['#f5'] is None, 'Q-95 F5 has no title bar')
f5 = page.rect('#f5')
page.page.mouse.move(f5[0] + 5, f5[1] + 3); page.page.mouse.down(); page.page.mouse.move(f5[0] + 105, f5[1] - 97, steps=8); page.page.mouse.up(); page.settle()
check(page.rect('#f5') == f5, 'Q-95 F5 cannot be dragged', page.rect('#f5'))
title = page.rect('#f4 > .win-title')
page.page.mouse.move(title[0] + 40, title[1] + 5); page.page.mouse.down(); page.page.mouse.move(title[0] - 60, title[1] + 55, steps=8); page.page.mouse.up(); page.settle()
moved = page.rect('#f4')
check(moved[:2] == [width - 252 - 100, 50], 'T-p77d a drag by the caption moves F4', moved)
page.page.mouse.click(page.rect('#form1')[0] + 300, 8); page.settle()
z = page.eval("() => Object.fromEntries(['#form1', '#f2', '#f3', '#f4', '#f5', '#f6'].map((s) => [s, Number(getComputedStyle(document.querySelector(s)).zIndex)]))")
check(min(z['#f2'], z['#f3']) > max(z['#form1'], z['#f4'], z['#f5'], z['#f6']), 'Q-71 F2 and F3 stay above the other windows, F5 does not', z)
page.eval("() => bausteine.act('fenster-anordnen')"); page.settle()
check(page.rect('#f4')[:2] == [width - 252, 0], 'Fenster anordnen puts F4 back where Form_Set puts it', page.rect('#f4'))
page.page.mouse.move(title[0] + 40, title[1] + 5); page.page.mouse.down(); page.page.mouse.move(title[0] - 60, title[1] + 55, steps=8); page.page.mouse.up(); page.settle()
page.page.reload()
page.page.wait_for_function("() => window.bausteine && bausteine.app.scene.id && document.documentElement.dataset.ready === '1'", timeout=60000)
page.settle()
check(page.rect('#f4')[:2] == [width - 252, 0], 'T-p77d a reload returns the window to the place of Form_Set', page.rect('#f4'))
lines = page.lines()
page.page.click('#f4 .win-close'); page.settle()
check(not page.visible('#f4') and page.eval("() => document.querySelector('[data-entry=\"fenster-zeigen\"]').disabled") is False, 'Q-85 the × of F4 hides it and enables Alle Fenster zeigen')
page.menu('ansicht/fenster-zeigen'); page.settle()
check(page.visible('#f4') and page.lines() == lines, 'Q-85 Alle Fenster zeigen shows it again with its state intact', page.lines())
page.page.click('#f3 .win-close'); page.page.click('#f2 .win-close'); page.settle()
page.page.click('#tb1 [data-tool="2"]') if page.visible('#f3') else None
page.eval("() => bausteine.act('fenster-zeigen')"); page.settle()
check(page.visible('#f2') and page.visible('#f3'), 'Q-85 and brings back F2 and F3')

## 16. The screen of 800 by 600

At 800 by 600 `Form_Set` makes F2 73 px high, room for one body, and a tool bar of Windows 98 shows
no scroll bar. A drag of the bottom edge makes the window taller, and the bar wraps into more
columns when the user widens it. F6 keeps its top of 580, so its top 20 px are on the screen, and
the user drags it into view. F5 covers the bottom 7 px of Form1. A second page opens at 800 by 600.

In [ ]:
small = Browser(site.url, width=800, height=600, look='classic')
small.open(screen='800x600')
VISIBLE = "() => { const r = document.querySelector('#f2 > .win-client').getBoundingClientRect(); const shown = [...document.querySelectorAll('#tb2 .tb-button')].filter((b) => { const q = b.getBoundingClientRect(); return q.top >= r.top - 0.5 && q.bottom <= r.bottom + 0.5 && q.right <= r.right + 0.5; }); return [shown.length, new Set(shown.map((b) => Math.round(b.getBoundingClientRect().left))).size]; }"
form1, f5 = small.rect('#form1'), small.rect('#f5')
check(form1 == [48, 0, 500, 550] and abs(form1[1] + form1[3] - f5[1] - 7) <= 1, 'Q-11 F5 covers the bottom 7 px of Form1', form1, f5)
check(small.rect('#f2') == [0, 500, 48, 73] and small.eval(VISIBLE) == [1, 1], 'Q-14 F2 is 73 px high and shows one body', small.rect('#f2'), small.eval(VISIBLE))
check(small.eval("() => [getComputedStyle(document.querySelector('#f2 > .win-client')).overflowY, getComputedStyle(document.querySelector('#tb2')).overflowY]") == ['hidden', 'visible'], 'Q-14 the bar shows no scroll bar')


def drag(selector, dx, dy):
    """Drags the middle of the element `selector` by dx and dy."""
    r = small.rect(selector)
    small.page.mouse.move(r[0] + r[2] / 2, r[1] + r[3] / 2); small.page.mouse.down()
    small.page.mouse.move(r[0] + r[2] / 2 + dx, r[1] + r[3] / 2 + dy, steps=6); small.page.mouse.up(); small.settle()


drag('#f2 > .win-edge[data-edge="s"]', 0, 20)
check(abs(small.rect('#f2')[3] - 93) <= 1, 'Q-14 a drag of the bottom edge makes F2 taller', small.rect('#f2'))
drag('#f2 > .win-edge[data-edge="n"]', 0, -120)
tall = small.eval(VISIBLE)
drag('#f2 > .win-edge[data-edge="e"]', 60, 0)
wide = small.eval(VISIBLE)
check(tall[1] == 1 and wide[1] == 2 and wide[0] > tall[0], 'Q-14 the bar wraps into more columns when F2 is widened', tall, wide)
small.eval("() => bausteine.act('fenster-anordnen')"); small.settle()
f6 = small.rect('#f6')
check(f6[1] == 580 and min(f6[1] + f6[3], 600) - f6[1] == 20, 'Q-15 F6 keeps its top of 580, so its top 20 px are on the screen', f6)
drag('#f6 > .win-title', 0, -300)
check(small.rect('#f6')[1] == 280 and small.rect('#f6')[1] + small.rect('#f6')[3] <= 600, 'Q-15 the user drags F6 into view', small.rect('#f6'))
front = small.eval("() => { const r = document.querySelector('#f6 .win-close').getBoundingClientRect(); const hit = document.elementFromPoint(r.left + r.width / 2, r.top + r.height / 2); return [hit !== null && hit.closest('#f6') !== null, getComputedStyle(document.querySelector('#f6')).zIndex, getComputedStyle(document.querySelector('#f4')).zIndex]; }")
expect(front[0], 'Q-15 the window dragged by its caption into view comes to the front, above F4', front)
small.eval("() => document.querySelector('#f6 .win-close').click()"); small.settle()
check(not small.visible('#f6'), 'the × of F6 hides it')
small.eval("() => bausteine.act('fenster-zeigen')"); small.settle()
check(small.visible('#f6'), 'Q-15 Alle Fenster zeigen brings F6 back after a close')
path, stats = small.screenshot('70_classic_800')
check(small.errors() == [], 'no error at 800 by 600', small.errors())
small.close()

## 17. The inverted views

Ansicht › Invertiert (Druck) turns the views white with black dots, as the thesis printed them, and
turns them back.

In [ ]:
r = page.rect('.view[data-view="1"]')
share = lambda colour: (lambda h: h.get(colour, 0) / sum(h.values()))(page.sample_box(r[0] + 4, r[1] + 4, r[2] - 8, r[3] - 8))
page.menu('ansicht/inverted'); page.settle()
check(share('#ffffff') > 0.8 and page.eval("() => bausteine.app.inverted") is True, 'Invertiert turns the views white', share('#ffffff'))
page.screenshot('70_classic_inverted')
page.menu('ansicht/inverted'); page.settle()
check(share('#000000') > 0.8, 'and back to black', share('#000000'))

## 18. The pan of the Schrägriss

A right or a middle drag in the classic Schrägriss moves the drawing with the pointer, as the right
and the middle drag pan a normal view. The worker keeps the pan in its state, the camera of the view
carries it in its translation, and the pick follows it. A left drag turns nothing in the classic
look and stays the click of 1.3, which zooms the 3D view in.

In [ ]:
#: Answers the lit pixels of the Schrägriss in the picture the worker reads back, the pan the worker keeps and the translation of the camera of the view.
VIEW_PIXELS = '''async () => {
  const b = bausteine.app.bridge;
  await b.render();
  const shot = await bausteine.snapshot();
  const cam = b.cameras;
  const w = cam[3 * 18 + 16], h = cam[3 * 18 + 17];
  const canvas = document.querySelector('#canvas').getBoundingClientRect();
  const r = document.querySelector('.view[data-view="3"]').getBoundingClientRect();
  const inset = Math.max(0, (r.width - w) / 2);
  const sx = shot.width / canvas.width;
  const x0 = Math.round((r.left + inset - canvas.left) * sx), y0 = Math.round((r.top + inset - canvas.top) * sx);
  const W = Math.round(w * sx), H = Math.round(h * sx);
  const px = new Uint8Array(shot.pixels.buffer, shot.pixels.byteOffset, shot.pixels.byteLength);
  const lit = [];
  for (let y = 0; y < H; y++) for (let x = 0; x < W; x++) {
    const i = ((y0 + y) * shot.width + x0 + x) * 4;
    if (px[i] + px[i + 1] + px[i + 2] > 60) lit.push([x, y]);
  }
  const state = await b.dump();
  return { lit, W, H, scale: sx, tx: cam[3 * 18 + 12], ty: cam[3 * 18 + 13], w, h, pan: state.obliquePan };
}'''
#: Answers for every item the id that the pick finds at the projection of its middle in the Schrägriss.
PICKS = '''async () => {
  const b = bausteine.app.bridge;
  const out = [];
  for (const it of bausteine.app.scene.items) {
    const p = b.project(3, it.position.x + it.size.x / 2, it.position.y + it.size.y / 2, it.position.z + it.size.z / 2);
    out.push([it.id, p ? await b.pick(3, p[0], p[1]) : null]);
  }
  return out;
}'''


def oblique_drag(by, button, steps=8):
    """Drags from the middle of the Schrägriss by `by` CSS pixels with the button `button`, in `steps` moves."""
    r = page.rect('.view[data-view="3"]')
    x, y = r[0] + r[2] / 2, r[1] + r[3] / 2
    page.page.mouse.move(x, y); page.page.mouse.down(button=button)
    for k in range(1, steps + 1):
        page.page.mouse.move(x + by[0] * k / steps, y + by[1] * k / steps)
    page.page.mouse.up(button=button); page.settle()


def shifted(a, b, d):
    """Answers the share of the lit pixels of `a` that are lit in `b` after a shift by `d`, among those that stay in the view."""
    lit = {tuple(p) for p in b['lit']}
    inside = [p for p in a['lit'] if 0 <= p[0] + d[0] < b['W'] and 0 <= p[1] + d[1] < b['H']]
    return sum((p[0] + d[0], p[1] + d[1]) in lit for p in inside) / len(inside) if inside else 0.0


fresh()
for x, y in ((0, 0), (0, 5)):
    page.page.click('#tb2 [data-body="Würfel"]'); page.settle()
    page.page.fill('#t1-1', str(x)); page.page.fill('#t1-2', str(y)); page.page.fill('#t1-4', '2')
    page.page.click('#btn-einf'); page.settle()
page.key('Escape'); page.key('Escape')
zoom = page.eval("() => bausteine.app.d3zoomf")
first = page.eval(VIEW_PIXELS)
check(first['pan'] == {'x': 0, 'y': 0} and len(first['lit']) > 200, 'the classic Schrägriss starts without a pan and draws its raster and its bodies', first['pan'], len(first['lit']))
oblique_drag((40, 24), 'right')
second = page.eval(VIEW_PIXELS)
check(second['pan'] == {'x': 40, 'y': 24} and page.eval("() => bausteine.app.d3zoomf") == zoom, 'Q-51 a right drag pans the classic Schrägriss, the worker keeps the pan, and the drag zooms nothing', second['pan'])
moved, kept = shifted(first, second, (round(40 * first['scale']), round(24 * first['scale']))), shifted(first, second, (0, 0))
check(moved > 0.99 and kept < 0.5, 'Q-51 the drawing after the right drag is the drawing before moved by the drag', moved, kept)
check(abs(second['tx'] - first['tx'] - 2 * 40 / first['w']) < 1e-4 and abs(second['ty'] - first['ty'] + 2 * 24 / first['h']) < 1e-4, 'Q-51 the camera of the Schrägriss carries the pan in its translation', first['tx'], second['tx'], first['ty'], second['ty'])
picks = page.eval(PICKS)
check(len(picks) == 2 and all(a == b for a, b in picks), 'Q-51 the pick follows the pan of the Schrägriss', picks)
oblique_drag((-48, -16), 'middle')
third = page.eval(VIEW_PIXELS)
followed = shifted(second, third, (round(-48 * first['scale']), round(-16 * first['scale'])))
check(third['pan'] == {'x': -8, 'y': 8} and followed > 0.99 and page.eval("() => bausteine.app.d3zoomf") == zoom, 'Q-51 a middle drag pans the classic Schrägriss as well', third['pan'], followed)
oblique_drag((50, 30), 'left')
check(page.eval(VIEW_PIXELS)['pan'] == third['pan'] and abs(page.eval("() => bausteine.app.d3zoomf") - zoom * 1.2) < 1e-9, 'Q-52 a left drag turns nothing in the classic look and stays the click of 1.3, which zooms the 3D view in', page.eval("() => bausteine.app.d3zoomf"))
page.page.click('#d3-zoomout'); page.settle()
oblique_drag((8, -8), 'middle', steps=2)
check(page.eval(VIEW_PIXELS)['pan'] == {'x': 0, 'y': 0} and page.eval("() => bausteine.app.d3zoomf") == zoom, 'a middle drag back puts the Schrägriss where it started', page.eval("() => bausteine.app.d3zoomf"))
check(page.errors() == [], 'no error on the way', page.errors())

## 19. The dialog Hilfe, the submenu Beispiele, Command1 and the hatch of FARBE

The classic look keeps the German quotation marks of the page POV-Referenz, which its reset of the
generated contents would remove. The dialog Hilfe keeps its height while its pages change, its tabs
form a property sheet of Windows 98 whose chosen tab stands taller, only the page scrolls, and the
samples keep their lines in a monospace font and scroll sideways. The arrows open and close the
submenu Beispiele as in the modern look. While the turn of tab Kamera runs, Command1 stays down as a
CommandButton held by the mouse, and a body with a texture hatches the frame FARBE with the brush of
Windows over its colour.

In [ ]:
page.eval("() => { bausteine.act('hilfe'); }"); page.settle()
#: Answers the box of the dialog Hilfe, the height of each tab, and for the dialog, its client and the page shown whether each scrolls.
HELP = '''() => {
  const d = document.querySelector('#help-dialog');
  const box = d.getBoundingClientRect();
  const tabs = [...d.querySelectorAll('.help-tab')].map((t) => ({ chosen: t.getAttribute('aria-selected') === 'true', height: t.getBoundingClientRect().height }));
  const scrolls = (e) => ({ overflow: getComputedStyle(e).overflowY, over: e.scrollHeight > e.clientHeight + 1 });
  return { box: [box.left, box.top, box.width, box.height], tabs, page: scrolls(d.querySelector('.help-page:not([hidden])')), dialog: scrolls(d), client: scrolls(d.querySelector('.help-client')) };
}'''
CALM = "() => document.querySelector('#help-dialog').getAnimations({ subtree: true }).length === 0"
pages = {}
for name in ('bedienung', 'tasten', 'pov'):
    page.page.click(f'#help-dialog [data-help-tab="{name}"]'); page.until(CALM)
    pages[name] = page.eval(HELP)
quotes = page.eval("() => { const q = document.querySelector('#help-dialog q.thesis'); return [getComputedStyle(q, '::before').content, getComputedStyle(q, '::after').content, getComputedStyle(q).quotes]; }")
check(quotes == ['open-quote', 'close-quote', '"„" "“" "‚" "‘"'], 'the classic page POV-Referenz keeps the German quotation marks that its reset would remove', quotes)
check(len({tuple(p['box']) for p in pages.values()}) == 1, 'the dialog Hilfe keeps its place and its height while the three pages change', {n: p['box'] for n, p in pages.items()})
heights = {n: [[t['chosen'], round(t['height'])] for t in p['tabs']] for n, p in pages.items()}
check(all(sorted(h, reverse=True) == [[True, 23], [False, 19], [False, 19]] for h in heights.values()), 'the chosen tab of the property sheet stands 23 px high against the 19 px of the others', heights)
check(all(p['page'] == {'overflow': 'auto', 'over': p['page']['over']} and not p['dialog']['over'] and not p['client']['over'] for p in pages.values()) and pages['pov']['page']['over'], 'only the page of the dialog Hilfe scrolls, the dialog and its client keep their size', pages)
code = page.eval("() => { const s = getComputedStyle(document.querySelector('#help-dialog pre.help-code')); return [s.overflowX, s.whiteSpace, s.fontFamily]; }")
check(code[:2] == ['auto', 'pre'] and code[2].endswith('monospace'), 'the samples keep their lines in a monospace font and scroll sideways', code)
page.key('Escape')
SUBMENU = "() => { const e = document.querySelector('#menubar [data-entry=beispiele]'); const m = document.querySelector('#menu-examples'); return { shown: getComputedStyle(m).display !== 'none', expanded: e.getAttribute('aria-expanded'), focus: document.activeElement === e ? 'entry' : m.contains(document.activeElement) ? 'inside' : 'elsewhere' }; }"
page.eval("() => document.activeElement && document.activeElement.blur()")
page.key('Alt+d')
for _ in range(12):
    if page.eval(SUBMENU)['focus'] == 'entry':
        break
    page.key('ArrowDown')
down = page.eval(SUBMENU)
page.key('ArrowRight')
right = page.eval(SUBMENU)
page.key('ArrowLeft')
left = page.eval(SUBMENU)
page.key('Escape'); page.key('Escape')
check(down == {'shown': False, 'expanded': 'false', 'focus': 'entry'}, 'the arrow down stops on Beispiele and leaves its submenu closed', down)
check(right == {'shown': True, 'expanded': 'true', 'focus': 'inside'}, 'the arrow right opens the submenu Beispiele with the focus inside', right)
check(left == {'shown': False, 'expanded': 'false', 'focus': 'entry'}, 'the arrow left closes it with the focus back on Beispiele', left)
page.page.click('#menubar .menu-item[data-menu="datei"] > .menu-title'); page.page.hover('#menubar [data-entry="beispiele"]'); page.settle()
hover = page.eval(SUBMENU)
page.key('Escape'); page.key('Escape')
check(hover['shown'] is True, 'the pointer still opens the submenu Beispiele', hover)
page.page.click('#sstab1 [data-tab="kamera"]'); page.settle()
page.page.fill('#hst-3', '90'); page.page.press('#hst-3', 'Enter'); page.settle()
idle = [page.css('#start', 'box-shadow'), page.css('#start', 'padding')]
page.page.click('#start')
page.until("() => { const s = document.querySelector('#start'); return s.getAttribute('aria-pressed') === 'true' && s.getAnimations().length === 0; }")
held = [page.css('#start', 'box-shadow'), page.css('#start', 'padding')]
page.page.click('#command2'); page.settle()
check(held[0].startswith('#000000 1px 1px 0px 0px inset') and held[1] == '2px 1px 0px 3px' and idle != held, 'Command1 stays down while the turn runs, with the pressed edge and the caption moved by one pixel', idle, held)
check(page.texts('#start', 'aria-pressed') == ['false'], 'Command2 releases Command1')
page.page.fill('#hst-3', '0'); page.page.press('#hst-3', 'Enter'); page.settle()
page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
page.page.click('#cb2 [data-line="1"]'); page.settle()
colour = page.eval("() => bausteine.app.scene.items[0].color").lower()
page.page.select_option('#cb3', 'Jade'); page.settle()
hatch = page.eval("() => { const f = document.querySelector('#frame-farbe'); const a = getComputedStyle(f, '::after'); return [f.hasAttribute('data-textured'), a.content, (a.maskImage || a.webkitMaskImage).split('(')[0]]; }")
check(page.eval("() => bausteine.app.scene.items[0].texture") == 'Jade' and hatch == [True, '""', 'url'], 'a body with a texture hatches the frame FARBE with the brush of Windows', hatch)
check(page.css('#frame-farbe', 'background-color') == colour, 'the hatched frame FARBE keeps the colour of the body', colour, page.css('#frame-farbe', 'background-color'))
page.key('Control+z')
check(page.eval("() => bausteine.app.scene.items[0].texture") is None and not page.eval("() => document.querySelector('#frame-farbe').hasAttribute('data-textured')"), 'Rückgängig takes the texture and the hatch away', page.eval("() => bausteine.app.scene.items[0].texture"))
check(page.errors() == [], 'no error on the way', page.errors())

## 20. Pictures and the end

A picture of the classic look at 1024 by 768, and the page logged no error on the way.

In [ ]:
path, stats = page.screenshot('70_classic_1024')
check(stats['lit'] > 0, 'the worker draws the classic views', stats)
check(page.errors() == [], 'no error on the way', page.errors())

In [ ]:
page.close()
site.stop()
check(DEFERRED == [], 'every check that waited for the end holds', *DEFERRED)